# Modelo 1: Predicción de Expectativa de Gol (xG) 
En este *notebook* vamos a construir el **Modelo 1** (Regresión Logística) para estimar la probabilidad de que un tiro sea gol (xG). Nos basaremos en el Feature Engineering previamente construido.

**Aclaración sobre las variables del biotipo del tiro**:
Las variables is_header, is_right_foot e is_left_foot se usan directamente en el modelo de regresión. Se extrajeron de la etiqueta qualifiers y fueron el paso intermedio para analizar estadísticamente la variable de texto body_part. Para el modelo (Regresión Logística con Scikit-Learn), es mucho mejor pasarle las variables numéricas de unos y ceros (las variables "is_something"), en lugar de pasarle la variable de texto "body_part", porque este algoritmo matemático puro de regresión necesita entradas en base numérica (One-hot encoding). 

In [ ]:
import os
# --- SOLUCIÓN DE RUTAS (CWD) 100% PORTABLE ---
# Este bloque ajusta el directorio de trabajo sin importar cómo se llame la carpeta raíz
if os.path.exists('CSV (API)') and os.path.exists('Modelo 1'):
    os.chdir('Modelo 1')
print('📁 Directorio configurado en:', os.getcwd())


In [ ]:
import ast
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos y Feature Engineering

In [ ]:
# Asumimos que la tabla ya tiene los qualifiers iniciales
df_shots = pd.read_csv('../CSV (API)/shots_with_qualifiers.csv')
df_shots = df_shots[(df_shots['x'] != 0) | (df_shots['y'] != 0)].copy()

# Parseo de banderas One-Hot
q = df_shots['qualifiers']
df_shots['is_big_chance'] = q.str.contains('BigChance', na=False).astype(int)
df_shots['is_header'] = q.str.contains('Head', na=False).astype(int)
df_shots['is_right_foot'] = q.str.contains('RightFoot', na=False).astype(int)
df_shots['is_left_foot'] = q.str.contains('LeftFoot', na=False).astype(int)
df_shots['is_penalty'] = q.str.contains('Penalty', na=False).astype(int)

# Distancia Euclidiana
df_shots['distance_to_goal'] = np.sqrt((100 - df_shots['x'])**2 + (50 - df_shots['y'])**2)

# Ángulo de Visión de Meta
goal_post_1_y = 54.8; goal_post_2_y = 45.2
x_dist = np.maximum(100 - df_shots['x'], 0.1)
angle_rad = np.arctan((goal_post_1_y - df_shots['y']) / x_dist) - np.arctan((goal_post_2_y - df_shots['y']) / x_dist)
df_shots['shot_angle_degrees'] = np.abs(angle_rad * (180 / np.pi))

print("Feature Engineering completado. Tamaño del dataset:", df_shots.shape)

# Parseo detallado de la zona del disparo
def parse_qualifiers(value):
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

def get_shot_zone(qualifiers):
    detailed_zones = {
        'BoxCentre', 'BoxLeft', 'BoxRight',
        'OutOfBoxCentre', 'OutOfBoxLeft', 'OutOfBoxRight',
        'SmallBoxCentre', 'SmallBoxLeft', 'SmallBoxRight'
    }
    for item in qualifiers:
        display = item.get('type', {}).get('displayName')
        if display in detailed_zones:
            return display
    for item in qualifiers:
        if item.get('type', {}).get('displayName') == 'Zone':
            return item.get('value', 'Unknown')
    return 'Unknown'

df_shots['qualifiers_parsed'] = df_shots['qualifiers'].apply(parse_qualifiers)
df_shots['shot_zone'] = df_shots['qualifiers_parsed'].apply(get_shot_zone)

zone_columns = [
    'zone_BoxCentre', 'zone_BoxLeft', 'zone_BoxRight',
    'zone_OutOfBoxCentre', 'zone_OutOfBoxLeft', 'zone_OutOfBoxRight',
    'zone_SmallBoxCentre', 'zone_SmallBoxLeft', 'zone_SmallBoxRight'
]

for col in zone_columns:
    zone_name = col.replace('zone_', '')
    df_shots[col] = (df_shots['shot_zone'] == zone_name).astype(int)


### 2. Selección de Variables y División Entrenamiento/Prueba
No entrenaremos con el 100% de los datos, separaremos un 20% exclusivamente para la prueba de métricas.

In [ ]:
# Seleccionar predictores
features = [
    'is_big_chance', 'is_penalty', 'distance_to_goal', 'shot_angle_degrees',
    'is_header', 'is_right_foot', 'is_left_foot',
    'zone_BoxCentre', 'zone_BoxLeft', 'zone_BoxRight',
    'zone_OutOfBoxCentre', 'zone_OutOfBoxLeft', 'zone_OutOfBoxRight',
    'zone_SmallBoxCentre', 'zone_SmallBoxLeft', 'zone_SmallBoxRight'
]
X = df_shots[features]
y = df_shots['is_goal']

# Eliminamos posibles NaNs por si acaso antes de entrenar
df_clean = df_shots.dropna(subset=features + ['is_goal'])
X = df_clean[features]
y = df_clean['is_goal']

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Datos de Entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de Prueba: {X_test.shape[0]} muestras")


### 3. Entrenamiento del Modelo de Regresión Logística

In [ ]:
numeric_features = ['distance_to_goal', 'shot_angle_degrees']
preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), numeric_features)],
    remainder='passthrough'
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, penalty='l2'))
])

model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] # Probabilidad de ser 1 (Gol)
print("¡Modelo entrenado exitosamente!")

### 4. Evaluación del Modelo y Métricas

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("--- MÉTRICAS EN EL SET DE PRUEBA (20%) ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC ROC:   {auc:.4f}")


# Agregar Baseline Naive
baseline_acc = 1 - y_test.mean()
print(f"Accuracy Baseline Naive (Predecir siempre 0): {baseline_acc:.4f}")
print(f"Mejora sobre el Baseline: {(acc - baseline_acc)*100:.2f}%")


#### 4.1 Matriz de Confusión Interactiva

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Predicción", y="Valor Real"),
                   x=['No Gol (0)', 'Gol (1)'],
                   y=['No Gol (0)', 'Gol (1)'],
                   title='Matriz de Confusión - Regresión Logística')
fig_cm.show()

#### Análisis
La **Matriz de Confusión** nos permite ver en detalle en qué se equivocó el modelo y en qué acertó evaluando con los datos de prueba nunca vistos:
* **Verdaderos Negativos (Cuadro arriba a la izquierda, No Gol -> No Gol)**: Cantidad de tiros que NO fueron gol y el modelo predijo correctamente que NO iban a ser gol. Constituyen la gran mayoría de los datos, ya que en el fútbol la mayoría de los tiros se fallan o los tapa el arquero.
* **Verdaderos Positivos (Cuadro abajo a la derecha, Gol -> Gol)**: Tiros que SÍ fueron gol y el modelo los identificó exitosamente por sus características mortales (cerca del arco, oportunidad clara, etc.).
* **Falsos Positivos (Cuadro arriba a la derecha, No Gol -> Gol)**: El modelo pensó que tenían altísima probabilidad de ser gol (xG muy alto), pero fallaron en la vida real. Esto pasa a menudo en el deporte cuando un jugador se 'come' un gol cantado o el arquero hace una atajada histórica.
* **Falsos Negativos (Cuadro abajo a la izquierda, Gol -> No Gol)**: El modelo les dio baja expectativa de gol, pero terminaron entrando mágicamente en la red. Son los famosos *'Golazos'* de larga distancia o desde ángulos casi imposibles.

#### 4.2 Curva ROC Interactiva

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'Regresión Logística (AUC = {auc:.4f})'))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash'), name='Aleatorio'))
fig_roc.update_layout(title='Curva ROC', xaxis_title='Tasa de Falsos Positivos (FPR)', yaxis_title='Tasa de Verdaderos Positivos (TPR)')
fig_roc.show()

#### Análisis
* La **Curva ROC** mide la capacidad general del modelo para distinguir diametralmente entre un 'Gol' y un 'No Gol' en diferentes umbrales de probabilidad.
* La **línea punteada diagonal** representa a un modelo mediocre que 'adivina' al azar (como lanzar una moneda 50% vs 50%). Todo el área que logremos capturar por encima de esa línea indica que nuestro modelo en verdad está aprendiendo a distinguir el peligro real de los tiros.
* El **AUC (Área Bajo la Curva)** es la calificación numérica de esta prueba: un valor de 1.0 significa perfección, mientras que 0.5 es adivinanza. Gracias al robusto *Feature Engineering* que definimos antes (identificando el `is_big_chance` y las ubicaciones (x,y)), nuestra curva azul traza un recorrido marcadamente empinado hacia la esquina superior izquierda, logrando solidificar el poder predictivo del **Expected Goals (xG)**.

### 5. Visualizaciones 3D Interactivas del xG
Gráfico 3D para entender cómo el modelo calcula la probabilidad en base a la Distancia y el Ángulo de un tiro.

In [ ]:
# Para la gráfica interactiva tomamos una muestra de los datos de testeo para que renderice fluido en el navegador web local
sample_idx = X_test.sample(n=min(1500, len(X_test)), random_state=42).index
X_sample = X_test.loc[sample_idx].copy()
y_sample_real = y_test.loc[sample_idx]
# Obtener la probabilidad de esos índices escogidos aleatoriamente del test set
val_indices = X_test.index.get_indexer(sample_idx)
probs_sample = y_prob[val_indices]

X_sample['prob_gol_xG'] = probs_sample
X_sample['is_goal_real'] = y_sample_real.apply(lambda x: 'Gol' if x==1 else 'No Gol')

fig_3d = px.scatter_3d(
    X_sample, 
    x='distance_to_goal', 
    y='shot_angle_degrees', 
    z='prob_gol_xG',
    color='is_goal_real',
    color_discrete_map={'Gol': 'green', 'No Gol': 'red'},
    opacity=0.7,
    title='Superficie de xG: Distancia vs Ángulo vs Probabilidad (Visualización 3D)',
    labels={'distance_to_goal': 'Distancia (m)', 'shot_angle_degrees': 'Ángulo', 'prob_gol_xG': 'xG (Prob. Gol)'}
)
# Ajustar la perspectiva inicial
fig_3d.update_layout(scene=dict(
        xaxis=dict(title='Distancia (m)'),
        yaxis=dict(title='Ángulo (grados)'),
        zaxis=dict(title='Probabilidad xG')
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig_3d.show()

#### Análisis
Este gráfico tridimensional es el resumen visual clave para interpretar nuestro modelo **xG**.

* **Eje X (Distancia)** y **Eje Y (Ángulo)**: Representan desde la espacialidad del terreno de juego dónde se efectuó el disparo respecto a la portería.
* **Eje Z (Probabilidad xG)**: Altura del punto. Es el porcentaje dictado por la regresión logística sobre las chances de que sea gol.
* **Colores**: Puntos verdes son los tiros que terminaron besando la red (Goles verdaderos), y puntos rojos son tiros atajados o fallados (Verdaderos Fallos).

**Conclusión Visual**: Al rotar la gráfica podemos notar inmediatamente que la densa masa de tiros se eleva astronómicamente a conformar probabilidades altas (picos en Z) a medida de que nos acercamos físicamente a cero metros (Eje X) y quedamos con un ángulo favorable y frontal (Eje Y). Interesantemente, es claro visualizar la presencia de puntos Rojos atrapados en la cúspide verde superior (Estos son Oportunidades Claras estúpidamente desperdiciadas), mientras que abajo en el llano rojo se detectan anómalos puntos Verdes solitarios (Los Golazos imposibles).

### 6. Conclusiones Definitivas del Desempeño del Modelo 

El modelo clasificador construido demuestra ser una herramienta robusta, conservadora y muy lógica. Analicemos su desempeño basándonos en las métricas obtenidas:

* **Accuracy (Exactitud = ~89.9%)**: El modelo acierta casi el 90% de las veces. Sin embargo, en el fútbol la inmensa mayoría de los tiros se fallan, por lo que el modelo "gana puntos fáciles" simplemente prediciendo casi siempre que un tiro **no** será gol. Por esto, miramos las demás métricas.

* **Precision (Precisión = ~68.1%)**: ¡Este es un número excelente y la mayor fortaleza del modelo! Significa que cuando el modelo se "arriesga" a predecir matemáticamente que un tiro terminará en Gol (1), acierta casi el 70% de las veces. Sabe identificar muy bien lo que es una verdadera amenaza o una oportunidad imperdonable de fallar.

* **Recall (Exhaustividad = ~18.6%)**: Aquí vemos la dificultad inherente del fútbol. El modelo solo logró "capturar" a priori un poco más del 18.6% de todos los goles que realmente ocurrieron. ¿Por qué es tan bajo? Porque la gran mayoría de los goles en la vida real provienen del caos absoluto (desvíos extraños, rebotes, errores del arquero, o golazos impensables de fuera del área) a los cuales el modelo lógicamente les había asignado una probabilidad muy baja de entrar (un xG bajo). El fútbol es impredecible por naturaleza.

* **F1-Score (~0.29)**: Al promediar la buena Precisión con el bajo Recall, obtenemos un F1 moderadamente bajo. Esto refleja perfectamente que el modelo es conservador frente al riesgo: prefiere no equivocarse diciendo inyectamente "Gol" (alta Precisión), aunque eso le cueste omitir los goles inexplicables y que le pasen por debajo del radar (bajo Recall).

* **Matriz de Confusión**: Confirma completamente esta postura conservadora frente al deporte. En la casilla de **Falsos Positivos** (tiros que creyó que eran gol y no entraron) el número es bajísimo y controlado, asegurando calidad al calificar un tiro de alto xG, mientras que en los **Falsos Negativos** (tiros que creyó que se iban a fallar y terminaron en golazo) hay una cifra amplia acumulando la imprecisión inherente al talento humano.

* **AUC ROC (~0.774)**: Esta es la métrica de oro, **la más importante** para probar un Expected Goals (xG). Un valor de AUC sobre 0.77 es bastante sólido considerando lo estocástico del fútbol. Se traduce textualmente en que si elegimos a ciegas un *Gol real* al azar y un *Tiro fallado* al azar, estamos un **77.4% seguros de que el modelo le había asignado un xG (Probabilidad Z de Plotly) más alta** al tiro que sí fue gol frente al que se falló. 

**Veredicto Final**: El trabajo logístico cumple con creces el objetivo. No se deja engañar fácilmente: penaliza y clasifica acertadamente las situaciones de peligro inminente (precisión contundente), y deja los goles épicos a cargo de la propia magia del deporte rey.